# RAG From Scratch: Production RAG, Freshness, Versions, and Security

Production RAG is less about another chain and more about controls: document versions, access control, freshness, citations, observability, and failure handling.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
from datetime import date

chunks = [
    {
        "id": "policy:v1:refunds",
        "text": "Refunds are available for 14 days.",
        "doc": "policy",
        "version": "v1",
        "valid_from": date(2025, 1, 1),
        "valid_to": date(2026, 1, 1),
        "allowed_roles": {"support", "admin"},
    },
    {
        "id": "policy:v2:refunds",
        "text": "Refunds are available for 30 days.",
        "doc": "policy",
        "version": "v2",
        "valid_from": date(2026, 1, 1),
        "valid_to": None,
        "allowed_roles": {"support", "admin"},
    },
    {
        "id": "security:v1:secrets",
        "text": "API keys must never be exposed in prompts or logs.",
        "doc": "security",
        "version": "v1",
        "valid_from": date(2025, 6, 1),
        "valid_to": None,
        "allowed_roles": {"admin"},
    },
]

In [ ]:
def is_valid(chunk, as_of):
    starts = chunk["valid_from"] <= as_of
    not_expired = chunk["valid_to"] is None or as_of < chunk["valid_to"]
    return starts and not_expired

def allowed(chunk, user_roles):
    return bool(chunk["allowed_roles"] & set(user_roles))

def retrieve_with_controls(query, user_roles, as_of):
    query_terms = set(query.lower().split())
    candidates = []
    for chunk in chunks:
        if not is_valid(chunk, as_of):
            continue
        if not allowed(chunk, user_roles):
            continue
        score = len(query_terms & set(chunk["text"].lower().replace(".", "").split()))
        if score:
            candidates.append((score, chunk))
    return [chunk for _, chunk in sorted(candidates, reverse=True, key=lambda item: item[0])]

retrieve_with_controls("refund policy", user_roles=["support"], as_of=date(2026, 6, 16))

In [ ]:
def build_answer_prompt(question, retrieved_chunks):
    citations = []
    context_lines = []
    for chunk in retrieved_chunks:
        citation = f"[{chunk['id']}]"
        citations.append(citation)
        context_lines.append(f"{citation} {chunk['text']}")
    return "\n".join([
        "Answer using only the context. Cite every factual claim with chunk ids.",
        "",
        "Context:",
        *context_lines,
        "",
        f"Question: {question}",
    ])

retrieved = retrieve_with_controls("How long are refunds available?", ["support"], date(2026, 6, 16))
print(build_answer_prompt("How long are refunds available?", retrieved))

Checklist for production: isolate tenant data, filter by ACL before generation, preserve versions and validity windows, redact secrets from traces, log retrieval decisions, evaluate on stale/conflicting documents, and require citations for factual claims.